# Milestone 1: Image Preprocessing, Enhancement, & Data Splitting
**System**: Skin Cancer Detection and Classification System Using CNN-SVM  
**Objective**: Resize, apply CLAHE contrast enhancement, normalize pixel values, perform leak-free lesion-level splitting, and augment training data.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from src.preprocessing import (
    apply_clahe,
    preprocess_single_image,
    augment_image_array,
    create_leak_free_splits,
    index_image_paths,
    CLASS_NAMES
)


## 1. Demonstrate CLAHE (Contrast Limited Adaptive Histogram Equalization)
Dermoscopic images often have uneven lighting and low lesion-to-skin contrast.  
CLAHE enhances local contrast in the L-channel of the LAB color space without altering colors.

In [ ]:
image_map = index_image_paths()
sample_id = list(image_map.keys())[0]
sample_path = image_map[sample_id]

raw_bgr = cv2.imread(sample_path)
raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
resized = cv2.resize(raw_rgb, (128, 128))
enhanced = apply_clahe(resized)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(resized)
axes[0].set_title('Original Resized (128x128)')
axes[0].axis('off')

axes[1].imshow(enhanced)
axes[1].set_title('CLAHE Enhanced (128x128)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 2. Demonstrate Data Augmentation
Augmentations include random horizontal flip, vertical flip, and rotations.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(enhanced)
axes[0].set_title('Original Base')
axes[0].axis('off')

for i in range(1, 5):
    aug = augment_image_array(enhanced)
    axes[i].imshow(aug)
    axes[i].set_title(f'Augmented #{i}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 3. Verify Leak-Free Lesion-Level Dataset Splitting

In [ ]:
metadata_path = 'dataset/raw/HAM10000_metadata.csv'
if not os.path.exists(metadata_path):
    metadata_path = 'HAM10000_metadata.csv'
df = pd.read_csv(metadata_path)
train_df, val_df, test_df = create_leak_free_splits(df)

print(f'Train: {len(train_df)} images ({train_df["lesion_id"].nunique()} lesions)')
print(f'Val:   {len(val_df)} images ({val_df["lesion_id"].nunique()} lesions)')
print(f'Test:  {len(test_df)} images ({test_df["lesion_id"].nunique()} lesions)')

# Check intersection
train_lesions = set(train_df['lesion_id'])
val_lesions = set(val_df['lesion_id'])
test_lesions = set(test_df['lesion_id'])
print('Train-Val overlap:', len(train_lesions.intersection(val_lesions)))
print('Train-Test overlap:', len(train_lesions.intersection(test_lesions)))
print('Val-Test overlap:', len(val_lesions.intersection(test_lesions)))
